# Baseline and Post-Iron-Infusion Hematologic, Iron Status, Redox, and Fibrinogen Measurements in Women with Iron-Deficiency Anemia Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.25ye-1ndm/fair2.json`

This dataset contains paired tabular biomedical measurements collected from women of reproductive age with iron-deficiency anemia at two timepoints:

- **T0**: baseline, before intravenous iron therapy
- **T1**: follow-up, 4–6 weeks after the final infusion

In this notebook, we will:
- load the Croissant metadata,
- inspect available record sets and fields,
- extract data into pandas DataFrames,
- perform simple preprocessing and exploratory analysis,
- compare baseline and follow-up measurements,
- and visualize selected numeric variables.

> **Important note**: all dataset entities should be referenced by their `@id` values whenever possible. Because Croissant metadata can evolve, the notebook includes discovery steps to inspect the actual available record set and field IDs before analysis.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn pyarrow

## 1. Data Loading
Load metadata from the dataset using `mlcroissant` and inspect the main descriptive fields.

We keep the Croissant schema URL in a variable so the notebook remains reusable and easy to adapt.

In [ ]:
import json
import math
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.25ye-1ndm/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON-serializable form
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset name:", metadata_json.get("name", "N/A"))
print("Description:", metadata_json.get("description", "N/A"))
print("Version:", metadata_json.get("version", "N/A"))
print("License:", metadata_json.get("license", "N/A"))
print("Published:", metadata_json.get("datePublished", "N/A"))
print("Identifier:", metadata_json.get("identifier", "N/A"))

In [ ]:
# A quick summary of selected descriptive metadata
summary_keys = [
    "name",
    "description",
    "citeAs",
    "temporalCoverage",
    "spatialCoverage",
    "dataCollection",
    "dataUseCases",
    "dataLimitations",
]

for key in summary_keys:
    value = metadata_json.get(key, None)
    print(f"\n{key}:")
    print(value)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The dataset package shown in the prompt suggests paired files and biomedical variables, but the reliable way to work with a Croissant dataset is to inspect the actual parsed metadata loaded by `mlcroissant`.

The next cells attempt to discover:
- record set IDs,
- field IDs inside each record set,
- and small record samples.

This helps us write robust analysis code even if field names differ slightly from expectations.

In [ ]:
# Inspect top-level metadata keys for orientation
print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

In [ ]:
# Inspect the recordSet section directly from metadata JSON
record_sets_meta = metadata_json.get("recordSet", [])
print(f"Number of record sets declared in metadata JSON: {len(record_sets_meta)}")

for i, rs in enumerate(record_sets_meta):
    print(f"\nRecord set #{i+1}")
    if isinstance(rs, dict):
        print("@id:", rs.get("@id"))
        print("name:", rs.get("name"))
        fields = rs.get("field", [])
        print("field count:", len(fields) if isinstance(fields, list) else "N/A")
    else:
        print(rs)

In [ ]:
# Utility function to safely inspect records from a record set by @id
def preview_records(ds, record_set_id, n=3):
    rows = []
    try:
        for idx, row in enumerate(ds.records(record_set=record_set_id)):
            rows.append(row)
            if idx + 1 >= n:
                break
    except Exception as e:
        print(f"Could not read records for {record_set_id}: {e}")
        return []
    return rows

# Try to preview records for each discovered record set
for rs in record_sets_meta:
    if isinstance(rs, dict):
        record_set_id = rs.get("@id")
        if record_set_id:
            print(f"\nPreviewing records for record set @id: {record_set_id}")
            sample_rows = preview_records(dataset, record_set_id, n=2)
            for row in sample_rows:
                print(row)

If the metadata contains record sets, the following cell builds a compact table of their `@id`s and field `@id`s. If no record sets are listed, that may indicate the Croissant file is minimal, externalized, or structured differently, and the notebook will fall back to inspection-based logic.

In [ ]:
# Build a record set -> field overview using @id references
recordset_overview = []

for rs in record_sets_meta:
    if not isinstance(rs, dict):
        continue
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    fields = rs.get("field", []) if isinstance(rs.get("field", []), list) else []
    field_ids = []
    for f in fields:
        if isinstance(f, dict):
            field_ids.append(f.get("@id"))
        else:
            field_ids.append(str(f))
    recordset_overview.append({
        "record_set_@id": rs_id,
        "record_set_name": rs_name,
        "n_fields": len(field_ids),
        "field_@ids": field_ids,
    })

overview_df = pd.DataFrame(recordset_overview)
overview_df

## 3. Data Extraction
Load data from each available record set into pandas DataFrames.

We use variables to dynamically handle record set IDs and keep everything based on Croissant `@id` references. The code is written to adapt whether the dataset has one or multiple record sets.

In [ ]:
# Collect record set @ids from metadata
record_set_ids = []
for rs in record_sets_meta:
    if isinstance(rs, dict) and rs.get("@id"):
        record_set_ids.append(rs.get("@id"))

print("Discovered record set @ids:")
print(record_set_ids)

In [ ]:
# Extract records from each record set into a DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

In [ ]:
# Display columns for each extracted DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nRecord set @id: {record_set_id}")
    print(f"Shape: {df.shape}")
    print("Columns:")
    print(df.columns.tolist())
    display(df.head())

### Optional fallback logic
If no record sets were discovered from metadata, the Croissant file may not expose `recordSet` entries directly in the parsed JSON form. The next cell provides a safe fallback by checking whether records can be accessed without specifying a record set, although most Croissant datasets require an explicit `record_set` argument.

In [ ]:
if not dataframes:
    print("No DataFrames were loaded from explicit record sets.")
    print("This can happen if the Croissant metadata does not expose recordSet entries directly.")
    print("Please inspect the Croissant JSON-LD schema manually or adapt this notebook once record set @ids are known.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as:
- identifying numeric fields,
- filtering records based on a threshold,
- normalizing a numeric variable,
- checking missing values,
- and grouping by a candidate categorical field if present.

Because Croissant datasets can use different field IDs, this notebook discovers suitable numeric and grouping columns dynamically from the loaded DataFrames.

In [ ]:
# Helper functions for flexible EDA
def get_numeric_columns(df):
    return df.select_dtypes(include=[np.number]).columns.tolist()

def get_candidate_group_columns(df, max_unique=10):
    candidates = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        if nunique > 1 and nunique <= max_unique:
            candidates.append(col)
    return candidates

def summarize_dataframe(df, name="DataFrame"):
    print(f"\n{name} summary")
    print("Shape:", df.shape)
    print("Missing values per column:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))
    numeric_cols = get_numeric_columns(df)
    if numeric_cols:
        display(df[numeric_cols].describe().T)
    else:
        print("No numeric columns detected.")

In [ ]:
# Summarize all loaded DataFrames
for record_set_id, df in dataframes.items():
    summarize_dataframe(df, name=record_set_id)

In [ ]:
# Select one record set for demonstration
# Preference: first loaded record set with at least one numeric column
selected_record_set_id = None
for rs_id, df in dataframes.items():
    if len(get_numeric_columns(df)) > 0:
        selected_record_set_id = rs_id
        break

if selected_record_set_id is None and dataframes:
    selected_record_set_id = list(dataframes.keys())[0]

print("Selected record set @id for detailed EDA:", selected_record_set_id)

In [ ]:
if selected_record_set_id is not None:
    selected_df = dataframes[selected_record_set_id].copy()
    numeric_fields = get_numeric_columns(selected_df)
    print("Numeric field @ids / column names:")
    print(numeric_fields)
else:
    selected_df = pd.DataFrame()
    numeric_fields = []

In [ ]:
# Choose a numeric field dynamically and perform filtering + normalization
if not selected_df.empty and numeric_fields:
    numeric_field_id = numeric_fields[0]
    numeric_series = selected_df[numeric_field_id].dropna()
    
    # Use median as a robust threshold instead of a hard-coded value
    threshold = numeric_series.median()
    filtered_df = selected_df[selected_df[numeric_field_id] > threshold].copy()
    
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Threshold (median): {threshold}")
    print(f"Filtered rows: {len(filtered_df)} / {len(selected_df)}")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
    else:
        filtered_df[f"{numeric_field_id}_normalized"] = 0.0
    
    print(f"Normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No suitable numeric field found for filtering and normalization.")

In [ ]:
# Try grouping by a low-cardinality field if one exists
if not selected_df.empty:
    group_candidates = get_candidate_group_columns(selected_df)
    print("Candidate grouping fields:")
    print(group_candidates)
    
    group_field_id = None
    for col in group_candidates:
        if col not in numeric_fields:
            group_field_id = col
            break
    
    if group_field_id is None and group_candidates:
        group_field_id = group_candidates[0]
    
    if group_field_id is not None and 'filtered_df' in globals() and not filtered_df.empty:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped filtered data by field: {group_field_id}")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found or filtered data is empty.")

### Outlier handling example
A common preprocessing step is to remove outliers using the interquartile range (IQR) rule. The next cell demonstrates this on the selected numeric field.

In [ ]:
if not selected_df.empty and numeric_fields:
    outlier_field_id = numeric_fields[0]
    series = selected_df[outlier_field_id].dropna()
    
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    no_outliers_df = selected_df[
        selected_df[outlier_field_id].between(lower_bound, upper_bound, inclusive="both")
    ].copy()
    
    print(f"Outlier handling for field: {outlier_field_id}")
    print(f"Lower bound: {lower_bound:.4f}")
    print(f"Upper bound: {upper_bound:.4f}")
    print(f"Original rows: {len(selected_df)}")
    print(f"Rows after outlier removal: {len(no_outliers_df)}")
else:
    print("No suitable numeric field available for outlier detection.")

## 5. Paired Comparison Preparation
The dataset description indicates paired baseline (`T0`) and follow-up (`T1`) measurements. If the Croissant metadata exposes separate record sets for each timepoint, we can compare them.

The next cells try to identify record sets corresponding to T0 and T1 using their `@id`, name, or column patterns. If such record sets are found, we prepare a simple merged comparison table.

In [ ]:
def find_timepoint_recordsets(dataframes_dict):
    t0_id, t1_id = None, None
    for rs_id, df in dataframes_dict.items():
        rs_id_lower = str(rs_id).lower()
        cols_lower = [str(c).lower() for c in df.columns]
        
        if ("t0" in rs_id_lower or "baseline" in rs_id_lower or "measurements_t0" in rs_id_lower) and t0_id is None:
            t0_id = rs_id
        if ("t1" in rs_id_lower or "follow" in rs_id_lower or "measurements_t1" in rs_id_lower) and t1_id is None:
            t1_id = rs_id
        
        # Weak fallback if there are exactly two record sets
    if (t0_id is None or t1_id is None) and len(dataframes_dict) == 2:
        keys = list(dataframes_dict.keys())
        if t0_id is None:
            t0_id = keys[0]
        if t1_id is None:
            t1_id = keys[1]
    return t0_id, t1_id

t0_record_set_id, t1_record_set_id = find_timepoint_recordsets(dataframes)
print("T0 record set @id:", t0_record_set_id)
print("T1 record set @id:", t1_record_set_id)

In [ ]:
# Attempt to identify a participant identifier column shared across both DataFrames
def find_common_id_column(df1, df2):
    common_cols = [col for col in df1.columns if col in df2.columns]
    priority_terms = ["id", "subject", "participant", "patient", "study"]
    for term in priority_terms:
        for col in common_cols:
            if term in str(col).lower():
                return col
    return common_cols[0] if common_cols else None

paired_df = None
merge_id_field = None

if t0_record_set_id in dataframes and t1_record_set_id in dataframes:
    t0_df = dataframes[t0_record_set_id].copy()
    t1_df = dataframes[t1_record_set_id].copy()
    merge_id_field = find_common_id_column(t0_df, t1_df)
    print("Candidate merge field @id / column name:", merge_id_field)
    
    if merge_id_field is not None:
        paired_df = pd.merge(
            t0_df,
            t1_df,
            on=merge_id_field,
            suffixes=("_t0", "_t1")
        )
        print("Merged paired DataFrame shape:", paired_df.shape)
        display(paired_df.head())
    else:
        print("Could not determine a shared participant identifier column.")
else:
    print("Could not identify both T0 and T1 record sets for paired comparison.")

### Change-score example
If we have matched T0 and T1 records and at least one shared numeric variable, we can compute a simple change score:

`change = value_t1 - value_t0`

In [ ]:
if paired_df is not None:
    t0_numeric_cols = [c for c in paired_df.columns if str(c).endswith("_t0") and pd.api.types.is_numeric_dtype(paired_df[c])]
    
    if t0_numeric_cols:
        base_col_t0 = t0_numeric_cols[0]
        base_name = base_col_t0[:-3]  # remove '_t0'
        base_col_t1 = f"{base_name}_t1"
        
        if base_col_t1 in paired_df.columns:
            change_field_id = f"{base_name}_change"
            paired_df[change_field_id] = paired_df[base_col_t1] - paired_df[base_col_t0]
            print(f"Computed paired change field: {change_field_id}")
            display(paired_df[[merge_id_field, base_col_t0, base_col_t1, change_field_id]].head())
            print(paired_df[change_field_id].describe())
        else:
            print("Matching T1 numeric field not found for change-score calculation.")
    else:
        print("No numeric _t0 columns found in the merged paired DataFrame.")
else:
    print("Paired DataFrame is not available.")

## 6. Visualization
Visualize selected field distributions and, when available, compare paired measurements between baseline and follow-up.

The notebook uses dynamically discovered numeric columns so it can remain useful even if exact field IDs differ across versions.

In [ ]:
# Histogram for one numeric field in the selected record set
if not selected_df.empty and numeric_fields:
    plot_field_id = numeric_fields[0]
    
    plt.figure(figsize=(8, 5))
    sns.histplot(selected_df[plot_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {plot_field_id}")
    plt.xlabel(plot_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for histogram.")

In [ ]:
# Boxplot for the same numeric field after outlier filtering, if available
if 'no_outliers_df' in globals() and not no_outliers_df.empty and 'outlier_field_id' in globals():
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=no_outliers_df[outlier_field_id].dropna())
    plt.title(f"Boxplot of {outlier_field_id} after IQR-based outlier removal")
    plt.xlabel(outlier_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("No outlier-filtered data available for boxplot.")

In [ ]:
# Correlation heatmap for up to 10 numeric columns in the selected DataFrame
if not selected_df.empty and len(numeric_fields) >= 2:
    corr_fields = numeric_fields[:10]
    corr = selected_df[corr_fields].corr(numeric_only=True)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
    plt.title("Correlation heatmap of selected numeric fields")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric fields available for a correlation heatmap.")

In [ ]:
# Paired scatter plot or before/after plot if a change field was computed
if paired_df is not None and 'base_col_t0' in globals() and 'base_col_t1' in globals() and base_col_t1 in paired_df.columns:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=paired_df[base_col_t0], y=paired_df[base_col_t1])
    
    min_val = np.nanmin([paired_df[base_col_t0].min(), paired_df[base_col_t1].min()])
    max_val = np.nanmax([paired_df[base_col_t0].max(), paired_df[base_col_t1].max()])
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red")
    
    plt.title(f"Paired comparison: {base_col_t0} vs {base_col_t1}")
    plt.xlabel(base_col_t0)
    plt.ylabel(base_col_t1)
    plt.tight_layout()
    plt.show()
else:
    print("Paired numeric comparison plot is not available.")

## 7. Conclusion
In this notebook, we:
- loaded a Croissant dataset with `mlcroissant`,
- accessed metadata through a single metadata object,
- inspected record sets and field structure using `@id` references,
- extracted records into pandas DataFrames,
- performed basic preprocessing such as filtering, normalization, missing-value review, and outlier handling,
- attempted a paired baseline/follow-up merge,
- and visualized numeric distributions and relationships.

### Key observations
- This dataset is designed for paired clinical analysis of women with iron-deficiency anemia before and after intravenous iron therapy.
- The most useful next steps are likely to include:
  - identifying exact field `@id`s for clinically important variables such as hemoglobin, ferritin, transferrin saturation, oxidative stress markers, and fibrinogen-related readouts,
  - computing within-participant changes between T0 and T1,
  - testing whether treatment is associated with systematic improvements in hematologic and biochemical markers,
  - and documenting field provenance directly from the Croissant schema.

### Recommendation
Before conducting inferential or biomedical analysis, review the full Croissant schema and confirm the exact `@id` values for:
- participant identifier fields,
- timepoint-specific record sets,
- and the target biomarker fields you intend to study.

That ensures all downstream analysis remains fully aligned with the dataset's formal metadata structure.